# Example 06: Which Measurements Matter Most?

## Interpolation Method Comparison & Measurement Sensitivity Analysis

This notebook explores the **spatial_interpolation** module of *soilactivity*,focusing on two critical questions for environmental monitoring campaigns:

> 1. **Which interpolation method should I use?** — Automated cross-validation>    comparison across multiple backends (RBF, Delaunay, IDW, Barnes, Cressman, etc.).
> 2. **Which measurements matter most?** — Leave-one-out sensitivity analysis>    that quantifies each measurement point's influence on the interpolated field.

The approach is inspired by **bssunfold**'s ``unfold_interpret`` module forunfold sensitivity diagnostics and the **pyoptexplain** pattern for explainingblack-box predictions through perturbation-based analysis.

We use synthetic Ambient Dose Equivalent Rate (ADER) data with three hot spotsto demonstrate method selection, critical point identification, and the impactof sparse versus dense measurement networks.


In [ ]:
sys.path.insert(0, '/home/z/my-project/soilactivity/src')
import numpy as np, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
warnings.filterwarnings('ignore')
import soilactivity as sa
from soilactivity.spatial_interpolation import (
    Interpolator2D, InterpolationAutoSelector,
    MeasurementSensitivityAnalyzer, AVAILABLE_METHODS)

plt.rcParams['figure.dpi'] = 120
print('soilactivity v' + sa.__version__)
print(str(len(AVAILABLE_METHODS)) + ' interpolation methods')


## Available Interpolation Methods

The ``spatial_interpolation`` module ships with 14 backends ranging fromfast nearest-neighbour lookups to Gaussian-Process regression with built-inuncertainty quantification. The ``AVAILABLE_METHODS`` catalogue below listsevery method and its description.


In [ ]:
print('{:25s} {}'.format('Method', 'Description'))
print('-' * 65)
for m, d in sorted(AVAILABLE_METHODS.items()):
    print('  {:25s} {}'.format(m, d))


## Synthetic ADER Data

We generate 100 measurement points randomly placed in a 1 km × 1 km areawith three Gaussian hot spots of varying intensity, plus a uniform backgroundof 0.1 µSv/h. Lognormal noise simulates realistic measurement uncertainty.

| Hot Spot | Centre (m) | Amplitude (µSv/h) | σ (m) |
|----------|------------|------------------------|--------|
| A        | (250, 300) | 2000                   | 80     |
| B        | (700, 650) | 1500                   | 100    |
| C        | (500, 200) | 1000                   | 60     |


In [ ]:
IMG = '/home/z/my-project/soilactivity/examples/'

np.random.seed(42)
N = 100
x = np.random.uniform(0, 1000, N)
y = np.random.uniform(0, 1000, N)

# Three Gaussian hot spots + background
spots = [
    (250, 300, 2000, 80),
    (700, 650, 1500, 100),
    (500, 200, 1000, 60),
]
ader = np.full(N, 0.1)  # 0.1 uSv/h background
for sx, sy, amp, sigma in spots:
    r2 = (x - sx)**2 + (y - sy)**2
    ader += amp * np.exp(-r2 / (2.0 * sigma**2))

# Add lognormal measurement noise (~10%)
noise_factor = np.random.lognormal(mean=0.0, sigma=0.1, size=N)
ader_noisy = ader * noise_factor

print('Points: {}'.format(N))
print('ADER range: {:.2f} - {:.2f} uSv/h'.format(ader_noisy.min(), ader_noisy.max()))
print('ADER median: {:.2f} uSv/h'.format(np.median(ader_noisy)))

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(x, y, c=ader_noisy, cmap='hot_r', s=40, edgecolors='k',
                linewidths=0.5, zorder=3)
fig.colorbar(sc, ax=ax, label='ADER (uSv/h)')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Synthetic ADER Measurements (100 points)')
ax.set_aspect('equal')
ax.set_xlim(0, 1000)
ax.set_ylim(0, 1000)
fig.tight_layout()
fig.savefig(IMG + 'fig06_data.png')
print('Saved fig06_data.png')


## Interpolation Comparison

The ``InterpolationAutoSelector`` runs k-fold cross-validation (k=5) acrosssix candidate methods and ranks them by RMSE.  We then visualise eachmethod's interpolated field side-by-side.


In [ ]:
candidates = ['rbf_tps', 'linear_delaunay', 'idw', 'barnes', 'cressman', 'nearest']
selector = InterpolationAutoSelector(candidates=candidates, cv_folds=5)
selector.fit(x, y, ader_noisy)
result = selector.select()

# Print ranking table
ranking = selector.get_ranking()
print('{:20s} {:>10s} {:>10s} {:>8s} {:>8s}'.format(
    'Method', 'RMSE', 'MAE', 'R^2', 'Time(s)'))
print('-' * 62)
for r in ranking:
    if 'error' in r:
        print('  {:18s}  FAILED: {}'.format(r['method'], r.get('notes', '')))
    else:
        print('  {:18s} {:10.4f} {:10.4f} {:8.4f} {:8.3f}'.format(
            r['method'], r['rmse'], r['mae'], r['r2'], r['time_s']))

# Recommendation
print()
print(selector.get_recommendation())

# --- 3x2 subplot of each method's field ---
xi_grid = np.linspace(0, 1000, 50)
yi_grid = np.linspace(0, 1000, 50)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()
vmin, vmax = ader_noisy.min(), ader_noisy.max()

for idx, method in enumerate(candidates):
    ax = axes[idx]
    try:
        interp = Interpolator2D(method=method)
        interp.fit(x, y, ader_noisy)
        Z, XI, YI = interp.predict_grid(xi_grid, yi_grid)
        pcm = ax.pcolormesh(XI, YI, Z, shading='auto', cmap='hot_r',
                           vmin=vmin, vmax=vmax)
        ax.scatter(x, y, c='k', s=6, zorder=3, alpha=0.5)
        ax.set_title(method, fontsize=10)
    except Exception as e:
        ax.text(0.5, 0.5, 'Error:\n' + str(e), transform=ax.transAxes,
                ha='center', va='center', fontsize=8)
    ax.set_aspect('equal')
    ax.set_xlim(0, 1000)
    ax.set_ylim(0, 1000)

fig.colorbar(pcm, ax=axes.tolist(), shrink=0.6, label='ADER (uSv/h)')
fig.suptitle('Interpolation Method Comparison', fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(IMG + 'fig06_comparison.png', bbox_inches='tight')
print('Saved fig06_comparison.png')

# --- Bar chart of RMSE ---
fig2, ax2 = plt.subplots(figsize=(8, 4))
valid_ranking = [r for r in ranking if 'error' not in r]
methods_bar = [r['method'] for r in valid_ranking]
rmses_bar = [r['rmse'] for r in valid_ranking]
colors_bar = ['steelblue'] * len(methods_bar)
if valid_ranking:
    best_idx = methods_bar.index(result['best_method'])
    colors_bar[best_idx] = 'crimson'
ax2.barh(methods_bar, rmses_bar, color=colors_bar, edgecolor='k')
ax2.set_xlabel('RMSE (uSv/h)')
ax2.set_title('Cross-validated RMSE by Method')
ax2.invert_yaxis()
fig2.tight_layout()
fig2.savefig(IMG + 'fig06_rmse_bar.png')
print('Saved fig06_rmse_bar.png')


## Sensitivity Analysis: Which Measurements Matter Most?

Inspired by **bssunfold**'s ``unfold_interpret`` and the **pyoptexplain**framework, we use ``MeasurementSensitivityAnalyzer`` to perform leave-one-outanalysis. Removing each point in turn and measuring the change in the interpolatedfield reveals which measurements are most *influential*.


In [ ]:
msa = MeasurementSensitivityAnalyzer()
msa.fit(x, y, ader_noisy, method='rbf_tps')

loo_results = msa.sensitivity_leave_one_out()
ranked = msa.ranking()
critical = msa.critical_points(90)

# Top-10 influential points table
print('Top-10 Most Influential Measurement Points')
print('=' * 72)
print('{:>5s} {:>8s} {:>8s} {:>10s} {:>12s} {:>12s} {:>12s}'.format(
    'Rank', 'x (m)', 'y (m)', 'ADER', 'Max Inf.', 'Mean Inf.', 'Area (km2)'))
print('-' * 72)
for i, r in enumerate(ranked[:10]):
    print('{:5d} {:8.1f} {:8.1f} {:10.2f} {:12.4f} {:12.4f} {:12.5f}'.format(
        i + 1, r['x'], r['y'], r['z'],
        r['max_influence'], r['mean_influence'], r['influence_area_km2']))

print()
print('Critical points (90th percentile): {} / {} points'.format(len(critical), N))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# (1) Scatter colored by max_influence with size proportional
ax1 = axes[0]
max_inf = np.array([r['max_influence'] for r in loo_results])
sizes = 20 + 200 * (max_inf / max_inf.max())
sc1 = ax1.scatter(x, y, c=max_inf, cmap='magma', s=sizes,
                    edgecolors='k', linewidths=0.5, zorder=3)
fig.colorbar(sc1, ax=ax1, label='Max Influence')
# Mark top-3 with red rings
for r in ranked[:3]:
    ax1.scatter(r['x'], r['y'], s=350, facecolors='none',
               edgecolors='lime', linewidths=2, zorder=4)
ax1.set_xlabel('x (m)')
ax1.set_ylabel('y (m)')
ax1.set_title('Point Influence (size ~ influence)')
ax1.set_aspect('equal')
ax1.set_xlim(0, 1000)
ax1.set_ylim(0, 1000)

# (2) Bar chart of top-10
ax2 = axes[1]
top10 = ranked[:10]
labels = ['P{}'.format(r['point_index']) for r in top10]
vals = [r['max_influence'] for r in top10]
bars = ax2.barh(labels, vals, color='orangered', edgecolor='k')
ax2.set_xlabel('Max Influence')
ax2.set_title('Top-10 Influential Points')
ax2.invert_yaxis()

# (3) Full vs without-top-3 difference map
ax3 = axes[2]
xi_grid = np.linspace(0, 1000, 50)
yi_grid = np.linspace(0, 1000, 50)

# Full interpolation
interp_full = Interpolator2D(method='rbf_tps')
interp_full.fit(x, y, ader_noisy)
Z_full, XI, YI = interp_full.predict_grid(xi_grid, yi_grid)

# Remove top-3 influential points
top3_idx = [r['point_index'] for r in ranked[:3]]
mask_keep = np.ones(N, dtype=bool)
mask_keep[top3_idx] = False
interp_reduced = Interpolator2D(method='rbf_tps')
interp_reduced.fit(x[mask_keep], y[mask_keep], ader_noisy[mask_keep])
Z_reduced, _, _ = interp_reduced.predict_grid(xi_grid, yi_grid)

diff = Z_full - Z_reduced
vabs = max(np.abs(diff).max(), 1e-10)
pcm = ax3.pcolormesh(XI, YI, diff, shading='auto', cmap='RdBu_r',
                      vmin=-vabs, vmax=vabs)
fig.colorbar(pcm, ax=ax3, label='ADER difference (uSv/h)')
ax3.scatter(x[top3_idx], y[top3_idx], c='lime', s=80, marker='x',
            linewidths=2, zorder=4, label='Removed')
ax3.scatter(x[mask_keep], y[mask_keep], c='grey', s=10, alpha=0.4, zorder=3)
ax3.legend(loc='upper right', fontsize=8)
ax3.set_xlabel('x (m)')
ax3.set_ylabel('y (m)')
ax3.set_title('Impact of Removing Top-3 Points')
ax3.set_aspect('equal')
ax3.set_xlim(0, 1000)
ax3.set_ylim(0, 1000)

fig.suptitle('Measurement Sensitivity Analysis', fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(IMG + 'fig06_sensitivity.png', bbox_inches='tight')
print('Saved fig06_sensitivity.png')


In [ ]:
print('=' * 60)
print('INTERPRETATION: What the Sensitivity Analysis Reveals')
print('=' * 60)
print()
# Identify where top-3 points sit relative to hot spots
print('Top-3 influential points location analysis:')
for r in ranked[:3]:
    nearest_spot = None
    min_dist = float('inf')
    for name, sx, sy, amp, sig in [('A', 250, 300, 2000, 80),
                                    ('B', 700, 650, 1500, 100),
                                    ('C', 500, 200, 1000, 60)]:
        d = np.sqrt((r['x'] - sx)**2 + (r['y'] - sy)**2)
        if d < min_dist:
            min_dist = d
            nearest_spot = name
    print('  Point {:3d} at ({:6.1f}, {:6.1f}) m, ADER={:7.2f} uSv/h,
          '  max_influence={:.4f}, nearest hot spot={}, dist={:.0f} m'.format(
              r['point_index'], r['x'], r['y'], r['z'],
              r['max_influence'], nearest_spot, min_dist))

print()
print('Key findings:')
print('  - Points near hot-spot centres have the highest influence')
print('    because they anchor the interpolation peak shape.')
print('  - Removing these points causes the largest deviations in the')
print('    interpolated field, especially near the hot-spot boundaries.')
print('  - The critical-points list identifies measurements that must be')
print('    retained to maintain interpolation fidelity.')
print('  - Points in flat (background) regions contribute least and could')
print('    be deprioritised in cost-constrained re-survey campaigns.')
print()
# Quantify total influence fraction
total_inf = sum(r['max_influence'] for r in ranked)
top3_inf = sum(r['max_influence'] for r in ranked[:3])
pct = 100.0 * top3_inf / total_inf if total_inf > 0 else 0
print('  The top-3 points account for {:.1f}% of total influence.'.format(pct))
print('  {} points are classified as critical (90th percentile).'.format(len(critical)))


## Sparse Interpolation

What happens when we only have 10 measurements instead of 100?  We comparethe full-data interpolation (100 pts) against a sparse sub-sample (10 randompts) and visualise the difference.


In [ ]:
np.random.seed(99)
sparse_idx = np.random.choice(N, size=10, replace=False)
x_sparse = x[sparse_idx]
y_sparse = y[sparse_idx]
z_sparse = ader_noisy[sparse_idx]

xi_grid = np.linspace(0, 1000, 50)
yi_grid = np.linspace(0, 1000, 50)

# Full interpolation
interp_full = Interpolator2D(method='rbf_tps')
interp_full.fit(x, y, ader_noisy)
Z_full, XI, YI = interp_full.predict_grid(xi_grid, yi_grid)

# Sparse interpolation
interp_sparse = Interpolator2D(method='rbf_tps')
interp_sparse.fit(x_sparse, y_sparse, z_sparse)
Z_sparse, _, _ = interp_sparse.predict_grid(xi_grid, yi_grid)

# Difference
Z_diff = Z_full - Z_sparse

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
vmin, vmax = ader_noisy.min(), ader_noisy.max()
vabs = max(np.nanmax(np.abs(Z_diff)), 1e-10)

# (1) Full (100 pts)
ax = axes[0]
pcm = ax.pcolormesh(XI, YI, Z_full, shading='auto', cmap='hot_r',
                      vmin=vmin, vmax=vmax)
ax.scatter(x, y, c='k', s=8, zorder=3, alpha=0.5)
fig.colorbar(pcm, ax=ax, label='ADER (uSv/h)')
ax.set_title('Full Network (100 pts)')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_aspect('equal')

# (2) Sparse (10 pts)
ax = axes[1]
pcm = ax.pcolormesh(XI, YI, Z_sparse, shading='auto', cmap='hot_r',
                      vmin=vmin, vmax=vmax)
ax.scatter(x_sparse, y_sparse, c='dodgerblue', s=50, edgecolors='k',
            linewidths=1, zorder=4)
fig.colorbar(pcm, ax=ax, label='ADER (uSv/h)')
ax.set_title('Sparse Network (10 pts)')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_aspect('equal')

# (3) Difference
ax = axes[2]
pcm = ax.pcolormesh(XI, YI, Z_diff, shading='auto', cmap='RdBu_r',
                      vmin=-vabs, vmax=vabs)
ax.scatter(x_sparse, y_sparse, c='k', s=30, marker='D', zorder=4)
fig.colorbar(pcm, ax=ax, label='Difference (uSv/h)')
ax.set_title('Full - Sparse Difference')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_aspect('equal')

for a in axes:
    a.set_xlim(0, 1000); a.set_ylim(0, 1000)

fig.suptitle('Sparse vs Dense Interpolation Comparison', fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(IMG + 'fig06_sparse.png', bbox_inches='tight')
print('Saved fig06_sparse.png')

# Quantify degradation
rmse_diff = np.sqrt(np.nanmean(Z_diff**2))
mae_diff = np.nanmean(np.abs(Z_diff))
print('Sparse vs Full  --  RMSE: {:.4f} uSv/h,  MAE: {:.4f} uSv/h'.format(rmse_diff, mae_diff))


## Conclusions

### Method Selection

- The ``InterpolationAutoSelector`` automatically identifies the best method  via cross-validation. For data with sharp hot spots, smooth RBF methods  (e.g. ``rbf_tps``) typically outperform nearest-neighbour and linear approaches.
- Barnes and Cressman methods provide good middle-ground performance with  physically motivated weighting functions.

### Critical Measurements

- Leave-one-out sensitivity analysis reveals that a small subset of measurements  (often those nearest to hot-spot peaks) dominate the interpolated field.
- The top 3 influential points can account for a large fraction of total influence,  meaning their removal would significantly degrade interpolation quality.
- Identifying critical points guides survey planning: ensure these locations are  measured with high precision and re-sampled if quality checks fail.

### Sparse Interpolation

- Reducing the network from 100 to 10 random points causes substantial  degradation in hot-spot reconstruction, with RMSE differences reflecting  lost spatial detail.
- A *targeted* sparse network (using sensitivity analysis to choose points)  would significantly outperform a random sub-sample.
- This underscores the value of adaptive sampling strategies in environmental  radiation monitoring campaigns.
